# ProQuest human source review
Run all cells to open the review. The original twelve pilot claims are preserved. Read each passage, answer the questions, then save and lock before revealing machine labels.

Source text and answers stay inside this workbench. **Do not download this notebook after running it.** Reopening it resumes saved responses. This does not run extraction, verification, or paid model calls.


In [ ]:
from pathlib import Path
import base64, hashlib, importlib, json, sys
PROJECT = Path('/home/ec2-user/SageMaker/election_arm')
if not PROJECT.is_dir():
    raise RuntimeError('Open this notebook inside the ProQuest TDM election_arm workbench.')
BUNDLE = PROJECT / 'source_review_pilot_tdm_v1'
BUNDLE.mkdir(exist_ok=True)
PAYLOAD = {'proquest_tdm_review.py': '...', 'pack.json': 'ewogICJzY2hlbWEiOiAicHJvcXVlc3QtcGlsb3QtdjEiLAogICJydWJyaWMiOiAiMjAyNi0xMC0xNC12MSIsCiAgInJlY29yZHMiOiBbCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTc1OjMyOTkiLAogICAgICAic291cmNlX3BhdGgiOiAiZGF0YS9wcm9xdWVzdC92ZXJpZmllZC9wcmVkX3Byb3F1ZXN0X2Vjb25vbXlfMTk3NS5leHBvcnQuanNvbmwiLAogICAgICAic291cmNlX3NoYTI1NiI6ICI2MGRhMTFmYzNmOTI1ZWNhMjQzOTU1YmEwOTA1NGRiOTIxOWIyODQ4NzA1MjAwZGYyNmU1NjZlYWUzYTMxNmVmIiwKICAgICAgInNvdXJjZV9saW5lIjogIjMyOTkiLAogICAgICAicGFnZV9pZCI6ICIxNTc4NzU2NTkiLAogICAgICAiZGF0ZSI6ICIxOTc1LTA5LTAzIiwKICAgICAgInB1Ymxpc2hlciI6ICJMb3MgQW5nZWxlcyBUaW1lcyAoMTkyMy0xOTk1KSIsCiAgICAgICJwaWxvdF9pbmNsdXNpb25fcHJvYmFiaWxpdHkiOiAiNi44MjM4MDM3MDE5MTM1MDhlLTA1IiwKICAgICAgInNvdXJjZV9sb2NhdG9yIjogIiIsCiAgICAgICJtYXBwaW5nX3N0YXR1cyI6ICJwZW5kaW5nIgogICAgfSwKICAgIHsKICAgICAgImF1ZGl0X2lkIjogInZlcmlmaWVkOjE5Nzg6MzAwIiwKICAgICAgInNvdXJjZV9wYXRoIjogImRhdGEvcHJvcXVlc3QvdmVyaWZpZWQvcHJlZF9wcm9xdWVzdF9lY29ub215XzE5NzguZXhwb3J0Lmpzb25sIiwKICAgICAgInNvdXJjZV9zaGEyNTYiOiAiMTYzNzhhZTA4YWE4ZWZmNGVhZmU0N2M5ODc5MDA4YjkyOGIwYmQzNGY0YTg2Nzk4NDg5YTU5MzU2ZWQ1MDJkMCIsCiAgICAgICJzb3VyY2VfbGluZSI6ICIzMDAiLAogICAgICAicGFnZV9pZCI6ICIxMjM1Mzk1NjYiLAogICAgICAiZGF0ZSI6ICIxOTc4LTEyLTMxIiwKICAgICAgInB1Ymxpc2hlciI6ICJOZXcgWW9yayBUaW1lcyAgKDE5MjMtKSIsCiAgICAgICJwaWxvdF9pbmNsdXNpb25fcHJvYmFiaWxpdHkiOiAiNi44MjM4MDM3MDE5MTM1MDhlLTA1IiwKICAgICAgInNvdXJjZV9sb2NhdG9yIjogIiIsCiAgICAgICJtYXBwaW5nX3N0YXR1cyI6ICJwZW5kaW5nIgogICAgfSwKICAgIHsKICAgICAgImF1ZGl0X2lkIjogInZlcmlmaWVkOjE5ODc6NDYxOSIsCiAgICAgICJzb3VyY2VfcGF0aCI6ICJkYXRhL3Byb3F1ZXN0L3ZlcmlmaWVkL3ByZWRfcHJvcXVlc3RfZWNvbm9teV8xOTg3LmV4cG9ydC5qc29ubCIsCiAgICAgICJzb3VyY2Vfc2hhMjU2IjogIjcwYzJmYmVjOTBhYzIwMjY2MGFhNTdkNWE3NzIxODRiOTcwNDViNjc2YTY3MTgxOGQ3YTVjYTk0OTBlMWRkN2MiLAogICAgICAic291cmNlX2xpbmUiOiAiNDYxOSIsCiAgICAgICJwYWdlX2lkIjogIjgxNDY3OTk5MCIsCiAgICAgICJkYXRlIjogIjE5ODctMDktMjgiLAogICAgICAicHVibGlzaGVyIjogIkxvcyBBbmdlbGVzIFRpbWVzICgxOTIzLTE5OTUpIiwKICAgICAgInBpbG90X2luY2x1c2lvbl9wcm9iYWJpbGl0eSI6ICI2LjgyMzgwMzcwMTkxMzUwOGUtMDUiLAogICAgICAic291cmNlX2xvY2F0b3IiOiAiIiwKICAgICAgIm1hcHBpbmdfc3RhdHVzIjogInBlbmRpbmciCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MTk4ODozOTEwIiwKICAgICAgInNvdXJjZV9wYXRoIjogImRhdGEvcHJvcXVlc3QvdmVyaWZpZWQvcHJlZF9wcm9xdWVzdF9lY29ub215XzE5ODguZXhwb3J0Lmpzb25sIiwKICAgICAgInNvdXJjZV9zaGEyNTYiOiAiODIxN2ZiOTgyMTNlOWVkN2Y4ZGIwOGZlNzgxNTQwYWEzMWVmYTVhNzNjMDIxZjdlYjhjYTQ2YzA3NmYxMmRjZCIsCiAgICAgICJzb3VyY2VfbGluZSI6ICIzOTEwIiwKICAgICAgInBhZ2VfaWQiOiAiOTAxNjQ2Mzk4IiwKICAgICAgImRhdGUiOiAiMTk4OC0wMy0xNiIsCiAgICAgICJwdWJsaXNoZXIiOiAiTG9zIEFuZ2VsZXMgVGltZXMgKDE5MjMtMTk5NSkiLAogICAgICAicGlsb3RfaW5jbHVzaW9uX3Byb2JhYmlsaXR5IjogIjYuODIzODAzNzAxOTEzNTA4ZS0wNSIsCiAgICAgICJzb3VyY2VfbG9jYXRvciI6ICIiLAogICAgICAibWFwcGluZ19zdGF0dXMiOiAicGVuZGluZyIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTkwOjM3MDAiLAogICAgICAic291cmNlX3BhdGgiOiAiZGF0YS9wcm9xdWVzdC92ZXJpZmllZC9wcmVkX3Byb3F1ZXN0X2Vjb25vbXlfMTk5MC5leHBvcnQuanNvbmwiLAogICAgICAic291cmNlX3NoYTI1NiI6ICI0MGVkZWFhZjZlYTBkM2FhMTI4MTdmNTk4ZGRiYzMxYzkwMDM3ZWM3ZDIzYjA5NzQ3YjA1MjkzNTJkNGQ4Njg0IiwKICAgICAgInNvdXJjZV9saW5lIjogIjM3MDAiLAogICAgICAicGFnZV9pZCI6ICIxNDU5ODIzMDExIiwKICAgICAgImRhdGUiOiAiMTk5MC0wNi0yMCIsCiAgICAgICJwdWJsaXNoZXIiOiAiTG9zIEFuZ2VsZXMgVGltZXMgKDE5MjMtMTk5NSkiLAogICAgICAicGlsb3RfaW5jbHVzaW9uX3Byb2JhYmlsaXR5IjogIjYuODIzODAzNzAxOTEzNTA4ZS0wNSIsCiAgICAgICJzb3VyY2VfbG9jYXRvciI6ICIiLAogICAgICAibWFwcGluZ19zdGF0dXMiOiAicGVuZGluZyIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTkwOjQ5ODQiLAogICAgICAic291cmNlX3BhdGgiOiAiZGF0YS9wcm9xdWVzdC92ZXJpZmllZC9wcmVkX3Byb3F1ZXN0X2Vjb25vbXlfMTk5MC5leHBvcnQuanNvbmwiLAogICAgICAic291cmNlX3NoYTI1NiI6ICI0MGVkZWFhZjZlYTBkM2FhMTI4MTdmNTk4ZGRiYzMxYzkwMDM3ZWM3ZDIzYjA5NzQ3YjA1MjkzNTJkNGQ4Njg0IiwKICAgICAgInNvdXJjZV9saW5lIjogIjQ5ODQiLAogICAgICAicGFnZV9pZCI6ICIxNDY3MDUzOTA5IiwKICAgICAgImRhdGUiOiAiMTk5MC0xMi0xOSIsCiAgICAgICJwdWJsaXNoZXIiOiAiTG9zIEFuZ2VsZXMgVGltZXMgKDE5MjMtMTk5NSkiLAogICAgICAicGlsb3RfaW5jbHVzaW9uX3Byb2JhYmlsaXR5IjogIjYuODIzODAzNzAxOTEzNTA4ZS0wNSIsCiAgICAgICJzb3VyY2VfbG9jYXRvciI6ICIiLAogICAgICAibWFwcGluZ19zdGF0dXMiOiAicGVuZGluZyIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTk4OjM0MDMiLAogICAgICAic291cmNlX3BhdGgiOiAiZGF0YS9wcm9xdWVzdC92ZXJpZmllZC9wcmVkX3Byb3F1ZXN0X2Vjb25vbXlfMTk5OC5leHBvcnQuanNvbmwiLAogICAgICAic291cmNlX3NoYTI1NiI6ICIxM2YwMzM5Y2Y5MDM2OTA1NDQwMDlhOGFjMjE2ZWE3ZGNkZDEzNGY3MTg5Y2MzYzIxOWQxOGYxYjFmZjMwOTA4IiwKICAgICAgInNvdXJjZV9saW5lIjogIjM0MDMiLAogICAgICAicGFnZV9pZCI6ICIyMTExNDk1NDk2IiwKICAgICAgImRhdGUiOiAiMTk5OC0xMC0wNiIsCiAgICAgICJwdWJsaXNoZXIiOiAiTG9zIEFuZ2VsZXMgVGltZXMgKDE5OTYtKSIsCiAgICAgICJwaWxvdF9pbmNsdXNpb25fcHJvYmFiaWxpdHkiOiAiNi44MjM4MDM3MDE5MTM1MDhlLTA1IiwKICAgICAgInNvdXJjZV9sb2NhdG9yIjogIiIsCiAgICAgICJtYXBwaW5nX3N0YXR1cyI6ICJwZW5kaW5nIgogICAgfSwKICAgIHsKICAgICAgImF1ZGl0X2lkIjogInZlcmlmaWVkOjIwMDE6NjM3NCIsCiAgICAgICJzb3VyY2VfcGF0aCI6ICJkYXRhL3Byb3F1ZXN0L3ZlcmlmaWVkL3ByZWRfcHJvcXVlc3RfZWNvbm9teV8yMDAxLmV4cG9ydC5qc29ubCIsCiAgICAgICJzb3VyY2Vfc2hhMjU2IjogIjkzMTcyOGNkZDU4ODkwOWI2N2M2NTEwOWVjYjkwNTAxNjgzYTZmOGVkNzYzMDQyYWQ4ODE4ZDBhZWRjMGU1OTkiLAogICAgICAic291cmNlX2xpbmUiOiAiNjM3NCIsCiAgICAgICJwYWdlX2lkIjogIjIxNTU3MzQxNzgiLAogICAgICAiZGF0ZSI6ICIyMDAxLTAzLTE2IiwKICAgICAgInB1Ymxpc2hlciI6ICJMb3MgQW5nZWxlcyBUaW1lcyAoMTk5Ni0pIiwKICAgICAgInBpbG90X2luY2x1c2lvbl9wcm9iYWJpbGl0eSI6ICI2LjgyMzgwMzcwMTkxMzUwOGUtMDUiLAogICAgICAic291cmNlX2xvY2F0b3IiOiAiIiwKICAgICAgIm1hcHBpbmdfc3RhdHVzIjogInBlbmRpbmciCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MjAwMToxMTE1OCIsCiAgICAgICJzb3VyY2VfcGF0aCI6ICJkYXRhL3Byb3F1ZXN0L3ZlcmlmaWVkL3ByZWRfcHJvcXVlc3RfZWNvbm9teV8yMDAxLmV4cG9ydC5qc29ubCIsCiAgICAgICJzb3VyY2Vfc2hhMjU2IjogIjkzMTcyOGNkZDU4ODkwOWI2N2M2NTEwOWVjYjkwNTAxNjgzYTZmOGVkNzYzMDQyYWQ4ODE4ZDBhZWRjMGU1OTkiLAogICAgICAic291cmNlX2xpbmUiOiAiMTExNTgiLAogICAgICAicGFnZV9pZCI6ICIzOTg3ODgzMzQiLAogICAgICAiZGF0ZSI6ICIyMDAxLTEwLTAxIiwKICAgICAgInB1Ymxpc2hlciI6ICJXYWxsIFN0cmVldCBKb3VybmFsIiwKICAgICAgInBpbG90X2luY2x1c2lvbl9wcm9iYWJpbGl0eSI6ICI2LjgyMzgwMzcwMTkxMzUwOGUtMDUiLAogICAgICAic291cmNlX2xvY2F0b3IiOiAiIiwKICAgICAgIm1hcHBpbmdfc3RhdHVzIjogInBlbmRpbmciCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MjAwMjo0OTIwIiwKICAgICAgInNvdXJjZV9wYXRoIjogImRhdGEvcHJvcXVlc3QvdmVyaWZpZWQvcHJlZF9wcm9xdWVzdF9lY29ub215XzIwMDIuZXhwb3J0Lmpzb25sIiwKICAgICAgInNvdXJjZV9zaGEyNTYiOiAiYzM0MWZhNjI5MGYwNzVhY2Q4ODU3ODY3MDIxYzZiM2ZkMDRjZjNiYjA0Njk3Yzc2ZGNmYTgyMmI3Njc4YzkyYyIsCiAgICAgICJzb3VyY2VfbGluZSI6ICI0OTIwIiwKICAgICAgInBhZ2VfaWQiOiAiMzA4NDk4NjcyIiwKICAgICAgImRhdGUiOiAiMjAwMi0wMS0zMSIsCiAgICAgICJwdWJsaXNoZXIiOiAiV2FsbCBTdHJlZXQgSm91cm5hbCIsCiAgICAgICJwaWxvdF9pbmNsdXNpb25fcHJvYmFiaWxpdHkiOiAiNi44MjM4MDM3MDE5MTM1MDhlLTA1IiwKICAgICAgInNvdXJjZV9sb2NhdG9yIjogIiIsCiAgICAgICJtYXBwaW5nX3N0YXR1cyI6ICJwZW5kaW5nIgogICAgfSwKICAgIHsKICAgICAgImF1ZGl0X2lkIjogInZlcmlmaWVkOjIwMDg6MTM0NDIiLAogICAgICAic291cmNlX3BhdGgiOiAiZGF0YS9wcm9xdWVzdC92ZXJpZmllZC9wcmVkX3Byb3F1ZXN0X2Vjb25vbXlfMjAwOC5leHBvcnQuanNvbmwiLAogICAgICAic291cmNlX3NoYTI1NiI6ICJlMjkzMmNiMjk1MTQxZDhmNDU5NDUyYzIyMTI5NjcyMjQwYTBmNTA0MmZmMjI1ZTk5ZmRmMGRjOGVkNTdhNWU3IiwKICAgICAgInNvdXJjZV9saW5lIjogIjEzNDQyIiwKICAgICAgInBhZ2VfaWQiOiAiNDA1NTY1MzYzIiwKICAgICAgImRhdGUiOiAiMjAwOC0xMi0wMyIsCiAgICAgICJwdWJsaXNoZXIiOiAiVGhlIENocmlzdGlhbiBTY2llbmNlIE1vbml0b3IiLAogICAgICAicGlsb3RfaW5jbHVzaW9uX3Byb2JhYmlsaXR5IjogIjYuODIzODAzNzAxOTEzNTA4ZS0wNSIsCiAgICAgICJzb3VyY2VfbG9jYXRvciI6ICIiLAogICAgICAibWFwcGluZ19zdGF0dXMiOiAicGVuZGluZyIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoyMDA5OjMyOTgiLAogICAgICAic291cmNlX3BhdGgiOiAiZGF0YS9wcm9xdWVzdC92ZXJpZmllZC9wcmVkX3Byb3F1ZXN0X2Vjb25vbXlfMjAwOS5leHBvcnQuanNvbmwiLAogICAgICAic291cmNlX3NoYTI1NiI6ICJiOGRkNGM1YWRiZmZhZDlhNWFhOTllY2RiYjM5ZGNiNmY2ZGVjNzRlNWEyNzkxZjlhZDc5M2I1ZDQzNWY3OGJlIiwKICAgICAgInNvdXJjZV9saW5lIjogIjMyOTgiLAogICAgICAicGFnZV9pZCI6ICIxOTI5Mjg4NDE1IiwKICAgICAgImRhdGUiOiAiMjAwOS0wNS0yOSIsCiAgICAgICJwdWJsaXNoZXIiOiAiTmV3IFlvcmsgVGltZXMgKDIwMDgtKSIsCiAgICAgICJwaWxvdF9pbmNsdXNpb25fcHJvYmFiaWxpdHkiOiAiNi44MjM4MDM3MDE5MTM1MDhlLTA1IiwKICAgICAgInNvdXJjZV9sb2NhdG9yIjogIiIsCiAgICAgICJtYXBwaW5nX3N0YXR1cyI6ICJwZW5kaW5nIgogICAgfQogIF0sCiAgImNvbXBhcmlzb25fc2hhMjU2IjogIjZkZWIyMmRlYzM0OTZiMmY5ZGUxMjU0MWRlZTZmM2FmMzk1OGI2NDAxNmNkZjNjNmE1NmY5NGY3N2FhNDc3YTMiLAogICJwYWNrX2lkIjogIjk0MmM0N2Y2ZjVlYmRlZDMwMDYyZGUyYzQwZDU0ZjRmYTM0MWZjOTJkYTRkMTU5MTUyYWJhNTIwMmQyNWQ4MGEiCn0K', 'comparison.json': 'ewogICJzY2hlbWEiOiAicHJvcXVlc3QtcGlsb3QtbGFiZWxzLXYxIiwKICAicmVjb3JkcyI6IFsKICAgIHsKICAgICAgImF1ZGl0X2lkIjogInZlcmlmaWVkOjE5NzU6MzI5OSIsCiAgICAgICJ0b3BpYyI6ICJnZW5lcmFsX2J1c2luZXNzIiwKICAgICAgImRpcmVjdGlvbiI6ICJ3b3JzZW4iLAogICAgICAic2NvcGUiOiAibmF0aW9uYWwiLAogICAgICAiY29uZmlkZW5jZSI6ICJoZWRnZWQiLAogICAgICAidm9pY2UiOiAiZXhwZXJ0IiwKICAgICAgImhvcml6b25fbW9udGhzIjogIjYiLAogICAgICAiaG9yaXpvbl9oaW50IjogInN0YXRlZCIsCiAgICAgICJsZWdhY3lfYmVuY2htYXJrIjogIklORFBSTyIsCiAgICAgICJsZWdhY3lfaG9yaXpvbl91c2VkIjogIjYuMCIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTc4OjMwMCIsCiAgICAgICJ0b3BpYyI6ICJnZW5lcmFsX2J1c2luZXNzIiwKICAgICAgImRpcmVjdGlvbiI6ICJ3b3JzZW4iLAogICAgICAic2NvcGUiOiAibmF0aW9uYWwiLAogICAgICAiY29uZmlkZW5jZSI6ICJoZWRnZWQiLAogICAgICAidm9pY2UiOiAiZXhwZXJ0IiwKICAgICAgImhvcml6b25fbW9udGhzIjogIjYiLAogICAgICAiaG9yaXpvbl9oaW50IjogInN0YXRlZCIsCiAgICAgICJsZWdhY3lfYmVuY2htYXJrIjogIklORFBSTyIsCiAgICAgICJsZWdhY3lfaG9yaXpvbl91c2VkIjogIjYuMCIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTg3OjQ2MTkiLAogICAgICAidG9waWMiOiAicHJpY2VzIiwKICAgICAgImRpcmVjdGlvbiI6ICJ3b3JzZW4iLAogICAgICAic2NvcGUiOiAibmF0aW9uYWwiLAogICAgICAiY29uZmlkZW5jZSI6ICJhc3NlcnRpdmUiLAogICAgICAidm9pY2UiOiAiam91cm5hbGlzdCIsCiAgICAgICJob3Jpem9uX21vbnRocyI6ICIxMiIsCiAgICAgICJob3Jpem9uX2hpbnQiOiAic3RhdGVkIiwKICAgICAgImxlZ2FjeV9iZW5jaG1hcmsiOiAiQ1BJIiwKICAgICAgImxlZ2FjeV9ob3Jpem9uX3VzZWQiOiAiMTIuMCIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTg4OjM5MTAiLAogICAgICAidG9waWMiOiAiZW1wbG95bWVudCIsCiAgICAgICJkaXJlY3Rpb24iOiAid29yc2VuIiwKICAgICAgInNjb3BlIjogIm5hdGlvbmFsIiwKICAgICAgImNvbmZpZGVuY2UiOiAiaGVkZ2VkIiwKICAgICAgInZvaWNlIjogIm9mZmljaWFsIiwKICAgICAgImhvcml6b25fbW9udGhzIjogInZhZ3VlIiwKICAgICAgImhvcml6b25faGludCI6ICJkZWZhdWx0IiwKICAgICAgImxlZ2FjeV9iZW5jaG1hcmsiOiAiVU5SQVRFIiwKICAgICAgImxlZ2FjeV9ob3Jpem9uX3VzZWQiOiAiMTIuMCIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoxOTkwOjM3MDAiLAogICAgICAidG9waWMiOiAiZ2VuZXJhbF9idXNpbmVzcyIsCiAgICAgICJkaXJlY3Rpb24iOiAid29yc2VuIiwKICAgICAgInNjb3BlIjogIm5hdGlvbmFsIiwKICAgICAgImNvbmZpZGVuY2UiOiAiaGVkZ2VkIiwKICAgICAgInZvaWNlIjogImV4cGVydCIsCiAgICAgICJob3Jpem9uX21vbnRocyI6ICJ2YWd1ZSIsCiAgICAgICJob3Jpem9uX2hpbnQiOiAiZGVmYXVsdCIsCiAgICAgICJsZWdhY3lfYmVuY2htYXJrIjogIklORFBSTyIsCiAgICAgICJsZWdhY3lfaG9yaXpvbl91c2VkIjogIjEyLjAiCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MTk5MDo0OTg0IiwKICAgICAgInRvcGljIjogImdlbmVyYWxfYnVzaW5lc3MiLAogICAgICAiZGlyZWN0aW9uIjogImltcHJvdmUiLAogICAgICAic2NvcGUiOiAibmF0aW9uYWwiLAogICAgICAiY29uZmlkZW5jZSI6ICJoZWRnZWQiLAogICAgICAidm9pY2UiOiAiZXhwZXJ0IiwKICAgICAgImhvcml6b25fbW9udGhzIjogIjEiLAogICAgICAiaG9yaXpvbl9oaW50IjogImRlZmF1bHQiLAogICAgICAibGVnYWN5X2JlbmNobWFyayI6ICJJTkRQUk8iLAogICAgICAibGVnYWN5X2hvcml6b25fdXNlZCI6ICIxLjAiCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MTk5ODozNDAzIiwKICAgICAgInRvcGljIjogImdlbmVyYWxfYnVzaW5lc3MiLAogICAgICAiZGlyZWN0aW9uIjogIndvcnNlbiIsCiAgICAgICJzY29wZSI6ICJuYXRpb25hbCIsCiAgICAgICJjb25maWRlbmNlIjogImhlZGdlZCIsCiAgICAgICJ2b2ljZSI6ICJleHBlcnQiLAogICAgICAiaG9yaXpvbl9tb250aHMiOiAiMTIiLAogICAgICAiaG9yaXpvbl9oaW50IjogInN0YXRlZCIsCiAgICAgICJsZWdhY3lfYmVuY2htYXJrIjogIklORFBSTyIsCiAgICAgICJsZWdhY3lfaG9yaXpvbl91c2VkIjogIjEyLjAiCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MjAwMTo2Mzc0IiwKICAgICAgInRvcGljIjogImdlbmVyYWxfYnVzaW5lc3MiLAogICAgICAiZGlyZWN0aW9uIjogIndvcnNlbiIsCiAgICAgICJzY29wZSI6ICJuYXRpb25hbCIsCiAgICAgICJjb25maWRlbmNlIjogImhlZGdlZCIsCiAgICAgICJ2b2ljZSI6ICJleHBlcnQiLAogICAgICAiaG9yaXpvbl9tb250aHMiOiAidmFndWUiLAogICAgICAiaG9yaXpvbl9oaW50IjogImRlZmF1bHQiLAogICAgICAibGVnYWN5X2JlbmNobWFyayI6ICJJTkRQUk8iLAogICAgICAibGVnYWN5X2hvcml6b25fdXNlZCI6ICIxMi4wIgogICAgfSwKICAgIHsKICAgICAgImF1ZGl0X2lkIjogInZlcmlmaWVkOjIwMDE6MTExNTgiLAogICAgICAidG9waWMiOiAiZ2VuZXJhbF9idXNpbmVzcyIsCiAgICAgICJkaXJlY3Rpb24iOiAid29yc2VuIiwKICAgICAgInNjb3BlIjogIm5hdGlvbmFsIiwKICAgICAgImNvbmZpZGVuY2UiOiAiYXNzZXJ0aXZlIiwKICAgICAgInZvaWNlIjogImV4cGVydCIsCiAgICAgICJob3Jpem9uX21vbnRocyI6ICIxMiIsCiAgICAgICJob3Jpem9uX2hpbnQiOiAic3RhdGVkIiwKICAgICAgImxlZ2FjeV9iZW5jaG1hcmsiOiAiSU5EUFJPIiwKICAgICAgImxlZ2FjeV9ob3Jpem9uX3VzZWQiOiAiMTIuMCIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoyMDAyOjQ5MjAiLAogICAgICAidG9waWMiOiAiZ2VuZXJhbF9idXNpbmVzcyIsCiAgICAgICJkaXJlY3Rpb24iOiAiaW1wcm92ZSIsCiAgICAgICJzY29wZSI6ICJuYXRpb25hbCIsCiAgICAgICJjb25maWRlbmNlIjogImFzc2VydGl2ZSIsCiAgICAgICJ2b2ljZSI6ICJvZmZpY2lhbCIsCiAgICAgICJob3Jpem9uX21vbnRocyI6ICJ2YWd1ZSIsCiAgICAgICJob3Jpem9uX2hpbnQiOiAiZGVmYXVsdCIsCiAgICAgICJsZWdhY3lfYmVuY2htYXJrIjogIklORFBSTyIsCiAgICAgICJsZWdhY3lfaG9yaXpvbl91c2VkIjogIjEyLjAiCiAgICB9LAogICAgewogICAgICAiYXVkaXRfaWQiOiAidmVyaWZpZWQ6MjAwODoxMzQ0MiIsCiAgICAgICJ0b3BpYyI6ICJnZW5lcmFsX2J1c2luZXNzIiwKICAgICAgImRpcmVjdGlvbiI6ICJ3b3JzZW4iLAogICAgICAic2NvcGUiOiAibmF0aW9uYWwiLAogICAgICAiY29uZmlkZW5jZSI6ICJhc3NlcnRpdmUiLAogICAgICAidm9pY2UiOiAiZXhwZXJ0IiwKICAgICAgImhvcml6b25fbW9udGhzIjogInZhZ3VlIiwKICAgICAgImhvcml6b25faGludCI6ICJkZWZhdWx0IiwKICAgICAgImxlZ2FjeV9iZW5jaG1hcmsiOiAiSU5EUFJPIiwKICAgICAgImxlZ2FjeV9ob3Jpem9uX3VzZWQiOiAiMTIuMCIKICAgIH0sCiAgICB7CiAgICAgICJhdWRpdF9pZCI6ICJ2ZXJpZmllZDoyMDA5OjMyOTgiLAogICAgICAidG9waWMiOiAiZ2VuZXJhbF9idXNpbmVzcyIsCiAgICAgICJkaXJlY3Rpb24iOiAiaW1wcm92ZSIsCiAgICAgICJzY29wZSI6ICJuYXRpb25hbCIsCiAgICAgICJjb25maWRlbmNlIjogImhlZGdlZCIsCiAgICAgICJ2b2ljZSI6ICJleHBlcnQiLAogICAgICAiaG9yaXpvbl9tb250aHMiOiAiMyIsCiAgICAgICJob3Jpem9uX2hpbnQiOiAiZGVmYXVsdCIsCiAgICAgICJsZWdhY3lfYmVuY2htYXJrIjogIklORFBSTyIsCiAgICAgICJsZWdhY3lfaG9yaXpvbl91c2VkIjogIjMuMCIKICAgIH0KICBdCn0K'}
decoded = {name: base64.b64decode(data) for name, data in PAYLOAD.items()}
for name, data in decoded.items():
    destination = BUNDLE / name
    if destination.exists() and destination.read_bytes() != data:
        raise RuntimeError('Existing installation differs: ' + name + '. Preserve responses and resolve version first.')
for name, data in decoded.items():
    destination = BUNDLE / name
    if not destination.exists():
        with destination.open('xb') as handle:
            handle.write(data)
sys.path.insert(0, str(BUNDLE))
import proquest_tdm_review
importlib.reload(proquest_tdm_review)
print('Ready. Responses remain in', BUNDLE / 'responses')


In [ ]:
review = proquest_tdm_review.launch(PROJECT, BUNDLE)
